# Jacobian smoke analysis plots

**Do not run this notebook on a NERSC login node or in Cursor on login** —
loading `jacobian_fields.h5` and the field galleries can OOM and drop the IDE.

Prefer the CPU script instead:

```bash
salloc -A m5326 -C cpu -q shared_interactive -t 00:30:00 -n 1 -c 8 --mem=32G
bash nersc_job_launchers/run_plot_jacobian_smoke_cpu.sh
```

Or: `python plotting/plot_jacobian_smoke.py`

Loads outputs from `/pscratch/sd/d/divyam/hine_runs/jacobian_smoke/` and writes PNGs to `plots/`.

- Scalars: `|λ|_max`, MSE, `‖e‖` (AR vs TF, all starts)
- Eps finite-difference check
- Field galleries: truth / AR / residual / TF / `|e_vec|`
- Mid/low AR latents for the first start

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np

OUT_DIR = Path("/pscratch/sd/d/divyam/hine_runs/jacobian_smoke")
PLOT_DIR = OUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

STEPS = [0, 5, 10, 19]
print("OUT_DIR:", OUT_DIR)
print("exists:", OUT_DIR.is_dir())
print("files:", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))

In [ ]:
summary_text = (OUT_DIR / "summary.txt").read_text()
print(summary_text)

traj = np.load(OUT_DIR / "trajectories.npz")
print("npz keys:", list(traj.keys()))
for k in traj.keys():
    a = traj[k]
    print(f"  {k}: shape={getattr(a, 'shape', None)} dtype={a.dtype}")

starts = traj["starts"]
file_indices = traj["file_indices"]
horizon = traj["ar_radius"].shape[1]
t = np.arange(horizon)
labels = [f"start={s} idx={int(fi)}" for s, fi in zip(starts, file_indices)]

## Scalar trajectories (AR solid, TF dashed)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
cmap = plt.cm.tab10(np.linspace(0, 1, len(starts), endpoint=False))

panels = [
    ("$|\\lambda|_{\\max}$", "ar_radius", "tf_radius"),
    ("MSE", "ar_mse", "tf_residual_mse"),
    (r"$\|e\|", "ar_pert_norm", "tf_pert_norm"),
]

for ax, (title, ar_key, tf_key) in zip(axes, panels):
    for i, (c, lab) in enumerate(zip(cmap, labels)):
        ax.plot(t, traj[ar_key][i], color=c, lw=2, label=lab)
        ax.plot(t, traj[tf_key][i], color=c, lw=1.5, ls="--", alpha=0.85)
    ax.set_xlabel("horizon step")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=8, loc="best")
axes[0].text(
    0.02, 0.02, "solid=AR, dashed=TF", transform=axes[0].transAxes,
    fontsize=8, va="bottom",
)
fig.suptitle("Jacobian smoke: scalars vs horizon", fontsize=12)
fig.savefig(PLOT_DIR / "scalars_vs_horizon.png", dpi=150)
plt.show()
print("wrote", PLOT_DIR / "scalars_vs_horizon.png")

## Finite-difference eps check

Spectral radius at the first start after warmup, for two FD step sizes. Large disagreement means the FD estimate is not eps-stable.

In [ ]:
eps = float(traj["eps"])
eps_check = float(traj["eps_check"])
r_eps = float(traj["radius_eps"])
r_eps_check = float(traj["radius_eps_check"])

print(f"radius(eps={eps:g}) = {r_eps:.6g}")
print(f"radius(eps={eps_check:g}) = {r_eps_check:.6g}")
print(f"ratio = {r_eps / max(r_eps_check, 1e-12):.4g}")

fig, ax = plt.subplots(figsize=(4.5, 3.5), constrained_layout=True)
ax.bar([f"eps={eps:g}", f"eps={eps_check:g}"], [r_eps, r_eps_check], color=["C0", "C1"])
ax.set_ylabel(r"$|\lambda|_{\max}$")
ax.set_title("FD eps check (first start, after warmup)")
ax.grid(True, axis="y", alpha=0.3)
fig.savefig(PLOT_DIR / "eps_check.png", dpi=150)
plt.show()
print("wrote", PLOT_DIR / "eps_check.png")

## Field galleries (one figure per start)

Rows: truth, AR pred, AR residual, TF pred, `|e_vec|` (AR).  
Columns: steps 0, 5, 10, 19. Color scale for vorticity fields shared from truth.

In [ ]:
def field2d(arr):
    """(1,H,W) or (H,W) -> (H,W)."""
    a = np.asarray(arr)
    return a[0] if a.ndim == 3 else a


def plot_start_gallery(h5, start_i, file_idx, steps=STEPS):
    truth = h5["truth_states"][start_i]  # (T+1, 1, H, W)
    ar = h5["autoreg_pred"][start_i]
    tf = h5["net_pred_truth"][start_i]
    res_ar = h5["residual_ar"][start_i]
    e_vec = h5["e_vec"][start_i]

    # Shared vorticity scale from truth over plotted steps (truth at t and t+1).
    truth_stack = np.concatenate([field2d(truth[s + 1])[None] for s in steps], axis=0)
    vmax = float(np.percentile(np.abs(truth_stack), 99))
    vmax = max(vmax, 1e-6)

    row_names = ["truth t+1", "AR pred", "AR residual", "TF pred", r"$|e_{AR}|$"]

    fig, axes = plt.subplots(
        len(row_names), len(steps),
        figsize=(3.0 * len(steps), 2.4 * len(row_names)),
        constrained_layout=True,
    )

    for j, s in enumerate(steps):
        panels = [
            (field2d(truth[s + 1]), "bwr", -vmax, vmax),
            (field2d(ar[s]), "bwr", -vmax, vmax),
            (field2d(res_ar[s]), "bwr", -vmax, vmax),
            (field2d(tf[s]), "bwr", -vmax, vmax),
            (np.abs(field2d(e_vec[s])), "magma", 0.0, None),
        ]
        for i, ((img, cmap, vmin, vmax_i), name) in enumerate(zip(panels, row_names)):
            ax = axes[i, j]
            if vmax_i is None:
                vmax_i = float(np.percentile(img, 99.5)) or 1e-6
                vmin = 0.0
            im = ax.imshow(img, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax_i, aspect="equal")
            if i == 0:
                ax.set_title(f"step {s}")
            if j == 0:
                ax.set_ylabel(name, fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
            if j == len(steps) - 1:
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle(f"start={int(starts[start_i])}  file_index={int(file_idx)}", fontsize=12)
    out = PLOT_DIR / f"fields_start{int(starts[start_i])}_idx{int(file_idx)}.png"
    fig.savefig(out, dpi=140)
    plt.show()
    print("wrote", out)


with h5py.File(OUT_DIR / "jacobian_fields.h5", "r") as h5:
    print("h5 keys:", list(h5.keys()))
    for i, fi in enumerate(file_indices):
        plot_start_gallery(h5, i, fi)

## Mid / low AR latents (first start)

In [ ]:
with h5py.File(OUT_DIR / "jacobian_fields.h5", "r") as h5:
    mid = h5["ar_mid"][0]   # (T, 1, 32, 32)
    low = h5["ar_low"][0]   # (T, 1, 8, 8)
    fi = int(file_indices[0])

    fig, axes = plt.subplots(2, len(STEPS), figsize=(3.0 * len(STEPS), 5.0), constrained_layout=True)
    for j, s in enumerate(STEPS):
        m = field2d(mid[s])
        l = field2d(low[s])
        vm = float(np.percentile(np.abs(m), 99)) or 1e-6
        vl = float(np.percentile(np.abs(l), 99)) or 1e-6
        im0 = axes[0, j].imshow(m, origin="lower", cmap="bwr", vmin=-vm, vmax=vm)
        im1 = axes[1, j].imshow(l, origin="lower", cmap="bwr", vmin=-vl, vmax=vl)
        axes[0, j].set_title(f"step {s}")
        for ax in (axes[0, j], axes[1, j]):
            ax.set_xticks([])
            ax.set_yticks([])
        if j == 0:
            axes[0, j].set_ylabel("AR mid (32×32)")
            axes[1, j].set_ylabel("AR low (8×8)")
        fig.colorbar(im0, ax=axes[0, j], fraction=0.046, pad=0.04)
        fig.colorbar(im1, ax=axes[1, j], fraction=0.046, pad=0.04)

    fig.suptitle(f"AR latents  start={int(starts[0])}  file_index={fi}")
    out = PLOT_DIR / f"latents_start{int(starts[0])}_idx{fi}.png"
    fig.savefig(out, dpi=140)
    plt.show()
    print("wrote", out)

print("All plots under", PLOT_DIR)